# F7 — what does a physical-size prune on the candidate list buy?

**Design of record: [`PREREGISTRATION.md`](PREREGISTRATION.md), including the §5a instrument amendment.**

One knob. The template-matching search, the candidate pool, the ranking, the padded ROI, the NMS
radius, the largest-CC template sizing and the seed draw are all frozen **by construction** — this
notebook never re-runs the search, it reads the pool `recall_workload_ledger.py` committed and
deletes candidates from it by measured physical size in µm.

Order of business:

1. the reproduction gate — does re-matching the *unpruned* cached pool reproduce the committed ledger?
2. instrument diagnostics — `on_nucleus`, residual merging, and what they cap
3. the three bound regimes: a-priori (4–18 µm), in-sample, and LODO
4. the pre-committed reading: LODO at 1.00 retention, paired ROI-median delta in depth-to-100%


In [1]:
import os, sys
import numpy as np, pandas as pd
from scipy.stats import binomtest

sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("."))
import f7_prune_eval as f7

pd.set_option("display.width", 200, "display.max_columns", 60)
gate, sizes, pool, ledger, ann = f7.load_all()
print(f"{len(gate)} cells, {gate.file_name.nunique()} ROIs, {gate.tumor_type.nunique()} domains")

70 cells, 14 ROIs, 7 domains


## 1. Reproduction gate

If this does not pass on all 70 cells, every number below is void: it would mean the cached
coordinates or the `gt_eval` reconstruction diverge from what produced the committed ledger.

The ledger's `rank` is **1-based** (`recall_workload_ledger.py` stores `ranks0 + 1`), so the gate
compares `recomputed_index + 1` against it.

In [2]:
print(f"ranks exact:            {int(gate.ranks_exact.sum())}/{len(gate)}")
print(f"n_gt_mitotic agrees:     {int((gate.n_gt_mitotic == gate.n_gt_mitotic_csv).sum())}/{len(gate)}")
assert bool(gate.ranks_exact.all()), "GATE FAILED -- stop"
assert bool((gate.n_gt_mitotic == gate.n_gt_mitotic_csv).all())

# the seed draw is with-replacement across seed indices, so some cells are duplicates
distinct = gate.groupby("file_name")["seed_ann_id"].nunique().rename("n_distinct_clicks")
print("\ndistinct clicks per ROI (5 seeds drawn with replacement):")
print(distinct.to_string())

ranks exact:            70/70
n_gt_mitotic agrees:     70/70

distinct clicks per ROI (5 seeds drawn with replacement):
file_name
013.tiff    5
094.tiff    4
201.tiff    4
233.tiff    3
245.tiff    5
246.tiff    5
300.tiff    4
301.tiff    5
402.tiff    5
403.tiff    4
459.tiff    5
460.tiff    5
529.tiff    4
548.tiff    5


## 2. Instrument diagnostics

Pre-committed in §5a: report `on_nucleus` and the residual-merging fraction with the results
either way. `on_nucleus` is the hard cap on what the size knob can do — candidates that land on no
segmented nucleus have an undefined size and are **kept**, never pruned.

In [3]:
diag = gate.groupby(["tumor_type", "file_name"]).agg(
    n_pool=("n_pool", "mean"), frac_on_nucleus=("frac_on_nucleus", "mean"),
    n_undefined=("n_size_undefined", "mean"), n_components=("n_components", "first")).round(3)
diag["max_prunable_frac"] = diag.frac_on_nucleus
print(diag.to_string())
print(f"\npooled on_nucleus = {gate.frac_on_nucleus.mean():.3f}"
      f"  (range {gate.frac_on_nucleus.min():.3f}-{gate.frac_on_nucleus.max():.3f})")

                                             n_pool  frac_on_nucleus  n_undefined  n_components  max_prunable_frac
tumor_type                       file_name                                                                        
canine cutaneous mast cell tumor 300.tiff   25741.4            0.395      15695.4         83661              0.395
                                 301.tiff   31208.8            0.406      19321.2         58264              0.406
canine lung cancer               201.tiff   20770.2            0.531       9861.8        115792              0.531
                                 233.tiff   27716.2            0.510      13951.0         90031              0.510
canine lymphosarcoma             245.tiff   38332.4            0.689      11996.4        100377              0.689
                                 246.tiff   30190.0            0.606      11857.6        166756              0.606
canine soft tissue sarcoma       459.tiff   22601.4            0.705       6704.

In [4]:
# residual merging: share of ON-NUCLEUS candidates in a component above nucleus_blobs'
# own validated max_area = 4000 px. Large values in the dense ROIs would mean an upper
# bound is measuring clumps rather than nuclei there.
rows = []
for _, g in gate.iterrows():
    a = sizes[f"{g.file_name}|{g.seed_index}|area_px"]
    on = ~np.isnan(a)
    rows.append(dict(file_name=g.file_name, tumor_type=g.tumor_type,
                     frac_gt_4000=float((a[on] > 4000).mean()) if on.any() else np.nan))
print(pd.DataFrame(rows).groupby(["tumor_type", "file_name"]).frac_gt_4000.mean().round(4).to_string())

tumor_type                        file_name
canine cutaneous mast cell tumor  300.tiff     0.0000
                                  301.tiff     0.0268
canine lung cancer                201.tiff     0.0967
                                  233.tiff     0.0710
canine lymphosarcoma              245.tiff     0.0026
                                  246.tiff     0.0534
canine soft tissue sarcoma        459.tiff     0.0525
                                  460.tiff     0.0000
human breast cancer               013.tiff     0.0282
                                  094.tiff     0.2978
human melanoma                    529.tiff     0.0191
                                  548.tiff     0.0788
human neuroendocrine tumor        402.tiff     0.2541
                                  403.tiff     0.2516


### What the sizes look like

Distribution of the measured equivalent diameter, TPs against the pool. This is a description of
the instrument's output, not the result — the result is what the prune *costs and saves*.

In [5]:
rows = []
for _, g in gate.iterrows():
    fn, si = g.file_name, int(g.seed_index)
    allv = sizes[f"{fn}|{si}|eqd_um"]; allv = allv[~np.isnan(allv)]
    tpv = f7.tp_sizes_for_cell(sizes, ledger, fn, si)
    rows.append(dict(file_name=fn, tumor_type=g.tumor_type, seed_index=si,
                     n_tp_measurable=len(tpv), tp_min=np.min(tpv) if len(tpv) else np.nan,
                     tp_p05=np.percentile(tpv, 5) if len(tpv) else np.nan,
                     tp_med=np.median(tpv) if len(tpv) else np.nan,
                     tp_p95=np.percentile(tpv, 95) if len(tpv) else np.nan,
                     tp_max=np.max(tpv) if len(tpv) else np.nan,
                     pool_med=np.median(allv), pool_p95=np.percentile(allv, 95)))
sz = pd.DataFrame(rows)
print(sz.groupby("file_name")[["n_tp_measurable", "tp_min", "tp_p05", "tp_med", "tp_p95",
                               "tp_max", "pool_med", "pool_p95"]].median().round(2).to_string())

           n_tp_measurable  tp_min  tp_p05  tp_med  tp_p95     tp_max  pool_med  pool_p95
file_name                                                                                
013.tiff              17.0    5.29    5.36    6.74   11.97  12.350000      7.45     14.84
094.tiff              72.0    4.14    4.95    6.15   17.73  26.219999     10.97     29.96
201.tiff              16.0    6.06    6.12    7.09   19.15  20.110001      8.57     20.87
233.tiff              17.0    4.24    4.27    7.00   15.13  22.360001      8.18     19.71
245.tiff              80.0    3.62    4.08    5.36    9.27  12.360000      5.75     11.59
246.tiff             108.0    3.64    4.22    5.59   12.71  25.160000      7.15     17.92
300.tiff             146.0    2.04    4.14    5.62    7.35   9.050000      5.82      8.49
301.tiff              21.0    2.20    3.87    5.33    6.94   9.960000      6.71     14.17
402.tiff             103.0    3.67    4.31    5.94   19.34  99.849998     10.73     27.53
403.tiff  

## 3. Run the three regimes

* **baseline** — no prune, the reference every delta is taken against
* **apriori** — `[4, 18] µm`, committed in §6 before any data was seen
* **insample@r** — bound from *this cell's own* TP sizes at retention `r`. Optimistic ceiling, not deployable.
* **lodo@r** — bound from the TP sizes of the **other six domains**. The deployment condition, and
  the only arm that speaks to "universal".

Re-matching after the prune is mandatory, not optional: `greedy_match` lets each annotation be
claimed by the best-ranked detection in range, so deleting a claimant can reassign or lose it.

In [6]:
# The arms, via the fast matcher (the radius query hoisted out of the per-arm loop).
# `verify_fast_path` asserts it equals `ev.bucket_detections` before any of it is used.
import time
for fn, si in [("013.tiff", 0), ("301.tiff", 2), ("459.tiff", 4)]:
    g = gate[(gate.file_name == fn) & (gate.seed_index == si)].iloc[0]
    cx, cy = pool[f"{fn}|{si}|cx"], pool[f"{fn}|{si}|cy"]
    for lo, hi in [(-np.inf, np.inf), (f7.APRIORI_LO, f7.APRIORI_HI)]:
        f7.verify_fast_path(cx, cy, sizes[f"{fn}|{si}|eqd_um"],
                            f7.gt_eval_for(ann, fn, int(g.seed_ann_id)),
                            float(g.match_radius_px), int(g.n_gt_mitotic), lo, hi)
print("fast matcher verified against ev.bucket_detections on 3 cells x 2 arms")

tps = {(g.file_name, int(g.seed_index)):
       f7.tp_sizes_for_cell(sizes, ledger, g.file_name, int(g.seed_index))
       for _, g in gate.iterrows()}
doms = gate.groupby("tumor_type")["file_name"].unique().to_dict()
lodo_pool = {d: np.concatenate([v for (f2, s2), v in tps.items() if f2 not in rn])
             for d, rn in doms.items()}

t1, rows = time.time(), []
for _, g in gate.iterrows():
    fn, si = g.file_name, int(g.seed_index)
    cx, cy = pool[f"{fn}|{si}|cx"], pool[f"{fn}|{si}|cy"]
    sz = sizes[f"{fn}|{si}|eqd_um"]
    ge = f7.gt_eval_for(ann, fn, int(g.seed_ann_id)); n_mit = int(g.n_gt_mitotic)
    neigh, gt_xy, det_xy = f7.cell_neighbours(cx, cy, ge, float(g.match_radius_px))
    gt_cls = ge["category_id"].to_numpy()
    base = dict(file_name=fn, tumor_type=g.tumor_type, seed_index=si, n_mit=n_mit,
                n_pool=int(g.n_pool), frac_on_nucleus=float(g.frac_on_nucleus))
    arms = [("baseline", -np.inf, np.inf, np.nan),
            ("apriori", f7.APRIORI_LO, f7.APRIORI_HI, np.nan)]
    for r in f7.QUANTS:
        arms.append((f"insample@{r:.2f}", *f7.bounds_from_tp_sizes(tps[(fn, si)], r), r))
        arms.append((f"lodo@{r:.2f}", *f7.bounds_from_tp_sizes(lodo_pool[g.tumor_type], r), r))
    for name, lo, hi, r in arms:
        rows.append({**base, "arm": name, "lo_um": lo, "hi_um": hi, "retention_nominal": r,
                     **f7.score_arm_fast(neigh, gt_xy, det_xy, gt_cls,
                                         f7.keep_mask(sz, lo, hi), n_mit)})
res = pd.DataFrame(rows)
res.to_csv(os.path.join(f7.RES, "f7_prune_results.csv"), index=False)
print(f"all arms in {time.time()-t1:.0f}s")
print(res.arm.value_counts().to_string())

fast matcher verified against ev.bucket_detections on 3 cells x 2 arms
all arms in 22s
arm
baseline         70
apriori          70
insample@1.00    70
lodo@1.00        70
insample@0.99    70
lodo@0.99        70
insample@0.98    70
lodo@0.98        70
insample@0.95    70
lodo@0.95        70


## 4. Candidate-list shortening and recall cost, by arm

`frac_kept` is the list shortening; `d_recall` is what it cost. Cell-level medians with the range,
then the ROI-level view.

In [7]:
summary = []
for arm in ["apriori"] + [f"insample@{r:.2f}" for r in f7.QUANTS] + [f"lodo@{r:.2f}" for r in f7.QUANTS]:
    t = f7.paired_table(res, arm)
    summary.append(dict(arm=arm,
        lo_um=t.lo_um.median(), hi_um=t.hi_um.median(),
        frac_kept_med=t.frac_kept.median(), frac_kept_min=t.frac_kept.min(),
        d_recall_med=t.d_recall.median(), d_recall_worst=t.d_recall.min(),
        cells_losing_tp=int((t.d_recall < 0).sum()), n_cells=len(t),
        d_depth90_med=t.d_depth_90.median(), d_depth95_med=t.d_depth_95.median(),
        d_depth100_med=t.d_depth_100.median()))
print(pd.DataFrame(summary).round(4).to_string(index=False))

          arm  lo_um   hi_um  frac_kept_med  frac_kept_min  d_recall_med  d_recall_worst  cells_losing_tp  n_cells  d_depth90_med  d_depth95_med  d_depth100_med
      apriori 4.0000 18.0000         0.9182         0.7675       -0.0080         -0.1176               39       70          131.0         1245.0           -22.0
insample@1.00 3.9776 20.1073         0.9327         0.7586        0.0000          0.0000                0       70          -64.0         -109.0          -283.5
insample@0.99 4.2032 19.7998         0.9285         0.7538       -0.0087         -0.1176               46       70          174.0          464.5          2463.0
insample@0.98 4.2434 14.1210         0.9120         0.7426       -0.0154         -0.1176               48       70          329.0          873.0          2676.5
insample@0.95 4.3552 12.7695         0.8752         0.6795       -0.0239         -0.1176               53       70          410.0         2790.0          5816.0
    lodo@1.00 0.4951 99.8467      

## 5. The pre-committed primary contrast

**LODO bound at 1.00 retention, paired ROI-median delta in depth-to-100%.** §8 also requires:
the saving must not be concentrated in the ROIs that were already cheap, and a sign flip between
domains kills the universality claim.

In [8]:
PRIMARY = "lodo@1.00"
t = f7.paired_table(res, PRIMARY)
roi = t.groupby(["tumor_type", "file_name"]).agg(
    n_pool=("n_pool", "median"), frac_kept=("frac_kept", "median"),
    d_recall=("d_recall", "median"), recall=("recall", "median"),
    depth90_base=("depth_90_base", "median"), depth90=("depth_90", "median"),
    depth95_base=("depth_95_base", "median"), depth95=("depth_95", "median"),
    depth100_base=("depth_100_base", "median"), depth100=("depth_100", "median"))
for q in (90, 95, 100):
    roi[f"d{q}"] = roi[f"depth{q}"] - roi[f"depth{q}_base"]
print(f"bound: [{t.lo_um.median():.2f}, {t.hi_um.median():.2f}] um\n")
print(roi.round(2).to_string())

bound: [0.50, 99.85] um

                                             n_pool  frac_kept  d_recall  recall  depth90_base  depth90  depth95_base  depth95  depth100_base  depth100    d90    d95    d100
tumor_type                       file_name                                                                                                                                   
canine cutaneous mast cell tumor 300.tiff   23524.0       1.00      0.00    1.00        2893.0   3255.0        5721.0   6368.0        11024.0   15266.0  362.0  647.0  4242.0
                                 301.tiff   32225.0       1.00      0.00    1.00        3179.0   3175.0        5876.0   5876.0        15769.0   17154.0   -4.0    0.0  1385.0
canine lung cancer               201.tiff   22631.0       1.00      0.00    1.00        1280.0   1280.0        2250.0   2250.0         2250.0    2250.0    0.0    0.0     0.0
                                 233.tiff   33221.0       1.00      0.00    1.00         434.0    434.0  

In [9]:
# ROI-level sign tests -- the ROI is the design unit, never the 70 cells (duplicate seeds)
print(f"arm = {PRIMARY}   bound [{t.lo_um.median():.2f}, {t.hi_um.median():.2f}] um\n")
for q in (90, 95, 100):
    d = roi[f"d{q}"].dropna()
    better = int((d < 0).sum()); n = len(d)
    print(f"depth_{q}: shorter on {better}/{n} ROIs, median delta {d.median():+.0f} candidates, "
          f"sign p={binomtest(better, n, 0.5).pvalue:.4f}"
          f"   [n={n}, p-floor {2/2**n:.4f}]")
print(f"\nROIs losing at least one mitosis: {int((roi.d_recall < 0).sum())}/{len(roi)}")
print(f"worst per-ROI recall change: {roi.d_recall.min():+.4f}")

arm = lodo@1.00   bound [0.50, 99.85] um

depth_90: shorter on 3/14 ROIs, median delta +0 candidates, sign p=0.0574   [n=14, p-floor 0.0001]
depth_95: shorter on 1/14 ROIs, median delta +0 candidates, sign p=0.0018   [n=14, p-floor 0.0001]
depth_100: shorter on 6/14 ROIs, median delta +0 candidates, sign p=0.7905   [n=14, p-floor 0.0001]

ROIs losing at least one mitosis: 1/14
worst per-ROI recall change: -0.0192


In [10]:
# is the saving concentrated in the ROIs that were already cheap?
c = roi[["depth100_base", "d100", "frac_kept"]].dropna().copy()
c["rel_saving"] = -c.d100 / c.depth100_base
print(c.sort_values("depth100_base").round(3).to_string())
print(f"\nSpearman(depth100_base, rel_saving) = "
      f"{c.depth100_base.corr(c.rel_saving, method='spearman'):+.3f}"
      f"   (negative => the prune pays least where the burden is worst)")

                                            depth100_base    d100  frac_kept  rel_saving
tumor_type                       file_name                                              
human breast cancer              013.tiff           366.0     0.0      0.999      -0.000
canine lung cancer               233.tiff          1328.0     0.0      0.998      -0.000
                                 201.tiff          2250.0     0.0      0.998      -0.000
human melanoma                   529.tiff          2516.0     0.0      0.998      -0.000
human breast cancer              094.tiff          4088.0     0.0      0.999      -0.000
human neuroendocrine tumor       403.tiff          5747.0   -69.0      0.990       0.012
canine lymphosarcoma             246.tiff          8502.0    -2.0      0.998       0.000
human neuroendocrine tumor       402.tiff         10028.0   778.0      0.981      -0.078
human melanoma                   548.tiff         10101.0    -3.0      0.999       0.000
canine soft tissue sa

### Does the bound transfer? (the universality check)

Per-fold LODO bounds, and whether any fold's bound would delete another fold's mitoses. A sign
flip or a non-overlapping bound between domains kills "universal" the same way it killed the
stain-residual criterion.

In [11]:
folds = []
tps = {(g.file_name, int(g.seed_index)): f7.tp_sizes_for_cell(sizes, ledger, g.file_name, int(g.seed_index))
       for _, g in gate.iterrows()}
doms = gate.groupby("tumor_type")["file_name"].unique().to_dict()
for d, roi_names in doms.items():
    held = np.concatenate([v for (f, s), v in tps.items() if f in roi_names])
    other = np.concatenate([v for (f, s), v in tps.items() if f not in roi_names])
    lo, hi = f7.bounds_from_tp_sizes(other, 1.00)
    folds.append(dict(domain=d, n_tp_held=len(held), lo_um=lo, hi_um=hi,
                      held_min=held.min(), held_max=held.max(),
                      frac_held_deleted=float(np.mean((held < lo) | (held > hi)))))
print(pd.DataFrame(folds).round(3).to_string(index=False))

                          domain  n_tp_held  lo_um  hi_um  held_min  held_max  frac_held_deleted
canine cutaneous mast cell tumor       1115  0.809 99.847     0.495  9.959000              0.004
              canine lung cancer        163  0.495 99.847     4.238 22.363001              0.000
            canine lymphosarcoma        880  0.495 99.847     1.815 25.156000              0.000
      canine soft tissue sarcoma        783  0.495 99.847     1.428 20.573999              0.000
             human breast cancer        452  0.495 99.847     4.142 33.658001              0.000
                  human melanoma       1218  0.495 99.847     2.823 23.327999              0.000
      human neuroendocrine tumor        564  0.495 33.658     0.809 99.847000              0.018


## 6. The whole retention curve

Reporting only the 1.00 level would repeat the residual probe's error — a min/max over ~100–240
TPs is an order statistic one atypical mitosis can pin.

In [12]:
curve = []
for r in f7.QUANTS:
    for kind in ("insample", "lodo"):
        t2 = f7.paired_table(res, f"{kind}@{r:.2f}")
        r2 = t2.groupby("file_name").median(numeric_only=True)
        curve.append(dict(regime=kind, retention=r,
                          lo_um=t2.lo_um.median(), hi_um=t2.hi_um.median(),
                          frac_kept=r2.frac_kept.median(),
                          d_recall_worst_roi=r2.d_recall.min(),
                          rois_losing_tp=int((r2.d_recall < 0).sum()),
                          d_depth90=r2.d_depth_90.median(),
                          d_depth95=r2.d_depth_95.median(),
                          d_depth100=r2.d_depth_100.median(),
                          rois_shorter_100=int((r2.d_depth_100 < 0).sum())))
cv = pd.DataFrame(curve)
cv.to_csv(os.path.join(f7.RES, "f7_retention_curve.csv"), index=False)
print(cv.round(4).to_string(index=False))

  regime  retention  lo_um   hi_um  frac_kept  d_recall_worst_roi  rois_losing_tp  d_depth90  d_depth95  d_depth100  rois_shorter_100
insample       1.00 3.9776 20.1073     0.9330              0.0000               0      -75.0     -117.0      -287.0                14
    lodo       1.00 0.4951 99.8467     0.9984             -0.0192               1        0.0        0.0        -1.5                 9
insample       0.99 4.2032 19.7998     0.9315             -0.1053              10      226.5      464.5       384.5                 3
    lodo       0.99 2.8234 24.6824     0.9741             -0.0192               1       -2.0        0.0       -32.5                 9
insample       0.98 4.2434 14.1210     0.9137             -0.1053              11      465.0     1154.0      4278.0                 2
    lodo       0.98 3.6196 22.2683     0.9599             -0.0192               4        9.5      138.0        -7.0                 8
insample       0.95 4.3552 12.7695     0.8834             -0.1

## 7. The a-priori bound

`[4, 18] µm`, committed in §6 from nuclear biology with no sight of the data. This is the honest
test of "universal by construction": it is the only bound that was not fitted to anything.

In [13]:
ta = f7.paired_table(res, "apriori")
ra = ta.groupby(["tumor_type", "file_name"]).median(numeric_only=True)
print(ra[["n_pool", "frac_kept", "recall_base", "recall", "d_recall",
          "depth_90_base", "depth_90", "depth_95_base", "depth_95",
          "depth_100_base", "depth_100"]].round(3).to_string())
for q in (90, 95, 100):
    d = ra[f"d_depth_{q}"].dropna(); b = int((d < 0).sum())
    print(f"\ndepth_{q}: shorter on {b}/{len(d)} ROIs, median {d.median():+.0f}, "
          f"p={binomtest(b, len(d), 0.5).pvalue:.4f}")
print(f"\nROIs losing >=1 mitosis: {int((ra.d_recall < 0).sum())}/{len(ra)}, "
      f"worst {ra.d_recall.min():+.4f}")
roi.to_csv(os.path.join(f7.RES, "f7_roi_primary.csv"))
ra.to_csv(os.path.join(f7.RES, "f7_roi_apriori.csv"))

                                             n_pool  frac_kept  recall_base  recall  d_recall  depth_90_base  depth_90  depth_95_base  depth_95  depth_100_base  depth_100
tumor_type                       file_name                                                                                                                                
canine cutaneous mast cell tumor 300.tiff   23524.0      0.960          1.0   0.989    -0.011         2893.0    4075.0         5721.0    8380.0         11024.0        NaN
                                 301.tiff   32225.0      0.968          1.0   1.000     0.000         3179.0    3206.0         5876.0    6138.0         15769.0    17034.0
canine lung cancer               201.tiff   22631.0      0.898          1.0   0.882    -0.118         1280.0    1185.0         2250.0       NaN          2250.0        NaN
                                 233.tiff   33221.0      0.918          1.0   1.000     0.000          434.0    1277.0         1328.0   17341.0  